In [1]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-english/crawl-300d-2M.vec.zip

--2024-02-10 06:52:01--  https://dl.fbaipublicfiles.com/fasttext/vectors-english/crawl-300d-2M.vec.zip
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.226.210.111, 13.226.210.25, 13.226.210.15, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.226.210.111|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1523785255 (1.4G) [application/zip]
Saving to: ‘crawl-300d-2M.vec.zip’

crawl-300d-2M.vec.z 100%[===================>]   1.42G  51.9MB/s    in 27s     

2024-02-10 06:52:29 (53.3 MB/s) - ‘crawl-300d-2M.vec.zip’ saved [1523785255/1523785255]



In [2]:
!unzip crawl-300d-2M.vec.zip

Archive:  crawl-300d-2M.vec.zip
  inflating: crawl-300d-2M.vec       


In [5]:
import numpy as np
import pandas as pd
from sklearn import preprocessing
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertTokenizer, BertModel
from transformers import RobertaTokenizer, RobertaModel
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, pairwise_distances
import matplotlib.pyplot as plt
import seaborn as sns
import gensim
from gensim.models import KeyedVectors

## Model Architecture
![model_architecture.png](./model_architecture.png)

In [6]:
ISEAR_CSV = "./datasets_csv/ISEAR_dataset.csv"
CLASS_LABELS = ['Anger', 'Disgust', 'Joy', 'Sadness']
ROBERTA_CONFIG = "roberta-base"
MAX_LEN = 256
BATCH_SIZE = 32
NUM_CLASSES = 4
LEARNING_RATE = 2e-5
EPOCHS = 10

In [29]:
class DualChannelModule(nn.Module):
    def __init__(self, input_size, lstm_hidden_size, dropout_val):
        super(DualChannelModule, self).__init__()
        self.channel1_bilstm = nn.LSTM(input_size=input_size, hidden_size=lstm_hidden_size, num_layers=1, batch_first=True, bidirectional=True)
        self.channel1_conv = nn.Conv1d(in_channels=lstm_hidden_size*2, out_channels=32, kernel_size=700)
        self.channel1_dropout = nn.Dropout(dropout_val)

        self.channel2_conv = nn.Conv1d(in_channels=input_size, out_channels=32, kernel_size=700)
        self.channel2_dropout = nn.Dropout(dropout_val)
        self.channel2_bilstm = nn.LSTM(input_size=32, hidden_size=lstm_hidden_size, num_layers=1, batch_first=True, bidirectional=True)

    def forward(self, roberta_ft_embeddings):

        roberta_ft_embeddings = roberta_ft_embeddings.permute(0, 2, 1)  # shape (BS, 1068, 256)
        channel1_bilstm_output, _ = self.channel1_bilstm(roberta_ft_embeddings)  # shape (BS, 1068, 32)
        channel1_bilstm_output = channel1_bilstm_output.permute(0, 2, 1)  # shape (BS, 32, 1068)
        channel1_conv_output = self.channel1_conv(channel1_bilstm_output)  # shape (BS, 32, 369)
        channel1_output = self.channel1_dropout(channel1_conv_output)  # shape (BS, 32, 369)

        roberta_ft_embeddings = roberta_ft_embeddings.permute(0, 2, 1)  # shape (BS, 256, 1068)
        channel2_conv_output = self.channel2_conv(roberta_ft_embeddings)  # shape (BS, 32, 369)
        channel2_dropout_output = self.channel2_dropout(channel2_conv_output)  # shape (BS, 32, 369)
        channel2_dropout_output = channel2_dropout_output.permute(0, 2, 1)  # shape (BS, 369, 32)
        channel2_output, _ = self.channel2_bilstm(channel2_dropout_output)  # shape (BS, 369, 32)
        channel2_output = channel2_output.permute(0, 2, 1)  # shape (BS, 32, 369)

        dual_channel_output = torch.cat((channel1_output, channel2_output), dim=2)  # shape (BS, 32, 738)
        return dual_channel_output


In [10]:
class ClassificationModule(nn.Module):
    def __init__(self, number_of_classes, dropout_val):
        super(ClassificationModule, self).__init__()
        self.maxpool = nn.MaxPool1d(kernel_size=2)
        self.dropout = nn.Dropout(dropout_val)
        self.flatten = nn.Flatten()
        self.dense1 = nn.Linear(in_features=11808, out_features=1000)
        self.dense2 = nn.Linear(in_features=1000, out_features=600)
        self.classification_output = nn.Linear(in_features=600, out_features=number_of_classes)

    def forward(self, dual_channel_output):
        maxpool_output = self.maxpool(dual_channel_output)  # shape (BS, 32, 369)
        dropout_output = self.dropout(maxpool_output)  # shape (BS, 32, 369)
        flatten_output = self.flatten(dropout_output)  # shape (BS, 11808)
        dense1_output = self.dense1(flatten_output)  # shape (BS, 1000)
        dense2_output = self.dense2(dense1_output)  # shape (BS, 600)
        classification_output = self.classification_output(dense2_output)  # shape (BS, 4)

        return classification_output

In [11]:
class TER(nn.Module):
    def __init__(self, number_of_classes, ft_model, input_size, lstm_hidden_size=16, dropout_val=0.1):
        super(TER, self).__init__()
        self.roberta = RobertaModel.from_pretrained(ROBERTA_CONFIG)
        self.fasttext = nn.Embedding.from_pretrained(torch.FloatTensor(ft_model.vectors), freeze=True)
        self.dual_channel = DualChannelModule(input_size=input_size, lstm_hidden_size=lstm_hidden_size, dropout_val=dropout_val)
        self.classifier = ClassificationModule(number_of_classes, dropout_val)

    def forward(self, input_seq, attention_mask):
        roberta_pooled_output = self.roberta(input_seq, attention_mask=attention_mask)[0]  # shape (BS, 256, 768)
        fasttext_output = self.fasttext(input_seq)  # shape (BS, 256, 300)
        roberta_fasttext_pooled_output = torch.cat([roberta_pooled_output, fasttext_output], axis=2)  # shape (BS, 256, 1068)
        dual_channel_output = self.dual_channel(roberta_fasttext_pooled_output)  # shape (BS, 32, 290)
        classification_output = self.classifier(dual_channel_output)  # shape (BS, 4)
        return classification_output

In [13]:
fasttext_model = KeyedVectors.load_word2vec_format('./crawl-300d-2M.vec')

In [14]:
embedding_dim = fasttext_model.vector_size
embedding_dim

300

In [15]:
def tokenize_sentences(sentences, tokenizer, max_length):
    tokenized_sequences = tokenizer(sentences, padding='max_length', truncation=True, max_length=max_length, return_tensors='pt')
    return tokenized_sequences['input_ids'], tokenized_sequences['attention_mask']

In [ ]:
'''def tokenize_sentences(sentences, tokenizer, fasttext_model, max_length):
    tokenized_sentences = [tokenizer(sentence, lower=True) for sentence in sentences]
    indexed_sentences = [
        [fasttext_model.key_to_index[word] for word in sentence if word in fasttext_model.key_to_index]
        for sentence in tokenized_sentences
    ]
    padded_sentences = [sentence + [0] * (max_length - len(sentence)) for sentence in indexed_sentences]
    train_input_ids = torch.tensor(padded_sentences)
    return train_input_ids'''

In [16]:
ISEAR_dataset = pd.read_csv(ISEAR_CSV, index_col=0)
ISEAR_dataset

,ISEAR_labels,ISEAR_sentences
0,joy,"During the period of falling in love, each tim..."
1,anger,When I was driving home after several days of...
2,sadness,When I lost the person who meant the most to me.
3,disgust,The time I knocked a deer down - the sight of ...
4,joy,When I got a letter offering me the Summer job...
...,...,...
4377,disgust,My roommate liked to listen to some meaningles...
4378,joy,I received a letter from a distant friend.
4379,anger,Two years back someone invited me to be the tu...
4380,sadness,I had taken the responsibility to do something...


In [17]:
ISEAR_train_data, ISEAR_val_data = train_test_split(ISEAR_dataset, test_size=0.1, random_state=42)
ISEAR_X_train = list(ISEAR_train_data['ISEAR_sentences'])
ISEAR_X_val = list(ISEAR_val_data['ISEAR_sentences'])
ISEAR_Y_train = list(ISEAR_train_data['ISEAR_labels'])
ISEAR_Y_val = list(ISEAR_val_data['ISEAR_labels'])

In [18]:
# Initialize XLM-RoBERTa tokenizer
tokenizer = RobertaTokenizer.from_pretrained(ROBERTA_CONFIG)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

In [19]:
train_input_ids, train_attention_mask = tokenize_sentences(ISEAR_X_train, tokenizer, max_length=MAX_LEN)
val_input_ids, val_attention_mask = tokenize_sentences(ISEAR_X_val, tokenizer, max_length=MAX_LEN)

In [20]:
#train_input_ids = tokenize_sentences(ISEAR_X_tra"in, tokenizer=gensim.utils.tokenize, fasttext_model=fasttext_model, max_length=MAX_LEN)
#val_input_ids = tokenize_sentences(ISEAR_X_val, tokenizer=gensim.utils.tokenize, fasttext_model=fasttext_model, max_length=MAX_LEN)

In [21]:
label_encoder = preprocessing.LabelEncoder()
ISEAR_Y_train_encoded = label_encoder.fit_transform(ISEAR_Y_train)
ISEAR_Y_val_encoded = label_encoder.fit_transform(ISEAR_Y_val)

In [22]:
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
label_mapping

{'anger': 0, 'disgust': 1, 'joy': 2, 'sadness': 3}

In [23]:
# Convert labels to PyTorch tensors
train_labels = torch.tensor(ISEAR_Y_train_encoded, dtype=torch.int64)
val_labels = torch.tensor(ISEAR_Y_val_encoded, dtype=torch.int64)

In [24]:
train_dataset = TensorDataset(train_input_ids, train_attention_mask, train_labels)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataset = TensorDataset(val_input_ids, val_attention_mask, val_labels)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

In [30]:
model = TER(number_of_classes=NUM_CLASSES, ft_model=fasttext_model, input_size=MAX_LEN)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [31]:
# Define optimizer and loss function
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

TER(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for batch_idx, (input_seq, attention_mask, labels) in enumerate(train_loader):
        input_seq = input_seq.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        output = model(input_seq, attention_mask)

        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if batch_idx % 50 == 49:
            print(f"Epoch [{epoch + 1}/{EPOCHS}] "
                  f"Batch [{batch_idx + 1}/{len(train_loader)}] "
                  f"Loss: {running_loss / 100:.4f}")
            running_loss = 0.0

Epoch [1/10] Batch [100/247] Loss: 1.0344
Epoch [1/10] Batch [200/247] Loss: 0.5964
Epoch [2/10] Batch [100/247] Loss: 0.4388
Epoch [2/10] Batch [200/247] Loss: 0.4337
Epoch [3/10] Batch [100/247] Loss: 0.2920
Epoch [3/10] Batch [200/247] Loss: 0.3202
Epoch [4/10] Batch [100/247] Loss: 0.2166
Epoch [4/10] Batch [200/247] Loss: 0.2369
Epoch [5/10] Batch [100/247] Loss: 0.1283
Epoch [5/10] Batch [200/247] Loss: 0.1428
Epoch [6/10] Batch [100/247] Loss: 0.1003
Epoch [6/10] Batch [200/247] Loss: 0.0869


In [ ]:
# Validation loop
model.eval()
total_correct = 0
total_samples = 0

predicted_labels = []
true_labels = []

with torch.no_grad():
    for input_seq, attention_mask, labels in val_loader:
        input_seq = input_seq.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        output = model(input_seq, attention_mask)
        _, predicted = torch.max(output, 1)

        total_correct += (predicted == labels).sum().item()
        total_samples += labels.size(0)

        predicted_labels.extend(predicted.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

# Calculate accuracy
accuracy = total_correct / total_samples
print(f"Validation Accuracy: {accuracy * 100:.2f}%")

# Calculate precision, recall, and F1-score
precision = precision_score(true_labels, predicted_labels, average='weighted')
recall = recall_score(true_labels, predicted_labels, average='weighted')
f1 = f1_score(true_labels, predicted_labels, average='weighted')

print(f"F1-score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

Validation Accuracy: 79.73%
F1-score: 0.7937
Precision: 0.8027
Recall: 0.7973


In [ ]:
cm = confusion_matrix(true_labels, predicted_labels, normalize="true")

In [ ]:
sns.set(font_scale=1.2)
sns.heatmap(cm, annot=True, fmt=".2f", cmap='Blues', cbar=True, xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS)

plt.title('Normalized Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
report = classification_report(true_labels, predicted_labels, target_names=CLASS_LABELS, output_dict=True)

print("{:<15} {:<15} {:<15} {:<15}".format("Class", "Precision", "Recall", "F1-Score"))
print("========================================================")
for class_label, metrics in report.items():
    if class_label in CLASS_LABELS:
        print("{:<15} {:<15.2f} {:<15.2f} {:<15.2f}".format(class_label, metrics['precision'], metrics['recall'], metrics['f1-score']))

print("\nOverall Accuracy: {:.2f}".format(report['accuracy']))